## 引导 + 配置

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "general" else Path.cwd()
sys.path.insert(0, str(ROOT))

from common.config import load_config, with_overrides
from common.utils import set_seed, ensure_dir

cfg = load_config("config.yaml")
cfg = with_overrides(cfg, epochs=3)
cfg

## 数据

In [ ]:
import torch
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader

def make_toy_data(cfg):
    X, y = make_classification(
        n_samples=cfg["n_samples"], n_features=cfg["n_features"],
        n_informative=10, n_classes=cfg["n_classes"],
        n_clusters_per_class=1, random_state=cfg["seed"]
    )
    X = X.astype("float32")
    X_tr, X_va, y_tr, y_va = train_test_split(
        X, y, test_size=0.2, random_state=cfg["seed"], stratify=y
    )
    return (torch.from_numpy(X_tr), torch.from_numpy(y_tr).long(),
            torch.from_numpy(X_va), torch.from_numpy(y_va).long())

def make_loaders(cfg):
    X_tr, y_tr, X_va, y_va = make_toy_data(cfg)
    bs, nw = int(cfg["batch_size"]), int(cfg["num_workers"])
    tr = DataLoader(TensorDataset(X_tr, y_tr), batch_size=bs, shuffle=True, num_workers=nw)
    va = DataLoader(TensorDataset(X_va, y_va), batch_size=bs, shuffle=False, num_workers=nw)
    return tr, va

## 模型

In [ ]:
import torch.nn as nn

def build_model(cfg) -> nn.Module:
    return nn.Sequential(
        nn.Linear(int(cfg["n_features"]), int(cfg["hidden_dim"])), nn.ReLU(), nn.Dropout(0.2),
        nn.Linear(int(cfg["hidden_dim"]), int(cfg["hidden_dim"])), nn.ReLU(),
        nn.Linear(int(cfg["hidden_dim"]), int(cfg["n_classes"]))
    )

## 训练主流程

In [ ]:
from common.checkpoint import init_early_stopping
from common.trainer import fit
from common.utils import get_device

def run_training(cfg, out_dir):
    device = get_device(cfg["device"])
    set_seed(int(cfg["seed"]))
    train_loader, val_loader = make_loaders(cfg)
    model = build_model(cfg).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=float(cfg["lr"]), weight_decay=float(cfg["weight_decay"]))
    criterion = nn.CrossEntropyLoss()
    es_state = init_early_stopping(int(cfg["patience"]), out_dir / "best.pt")
    model, history, es_state = fit(
        model, train_loader, val_loader, optimizer, criterion, device,
        epochs=int(cfg["epochs"]), es_state=es_state
    )
    return model, history, es_state

out_dir = ensure_dir(cfg["output_dir"])
model, history, es_state = run_training(cfg, out_dir)
es_state

## 画图 + 推理闭环

In [ ]:
from common.plot import plot_history
from common.checkpoint import load_checkpoint

plot_history(history, out_dir / "curves.png")

@torch.no_grad()
def predict(model, X, device):
    model.eval()
    return model(torch.as_tensor(X, dtype=torch.float32).to(device)).argmax(1).cpu().numpy()

device = get_device(cfg["device"])
best = load_checkpoint(build_model(cfg), out_dir / "best.pt", device)
print(predict(best, torch.randn(5, int(cfg["n_features"])), device))